In [ ]:
import pandas as pd
import subprocess
subprocess.run(['pip', 'install', 'featuretools'], capture_output=True)
import featuretools as ft


In [ ]:
# Read in undersampled data
dtypes = {
    'ip':           'uint32',
    'app':          'uint16',
    'device':       'uint16',
    'os':           'uint16',
    'channel':      'uint16',
    'is_attributed':'uint8',
}
df = pd.read_csv('train.csv', dtype=dtypes, parse_dates=['click_time', 'attributed_time'])

# Keep all legitimate clicks + undersample fraud to reach 20M total
legit        = df[df['is_attributed'] == 1]
fraud_sample = df[df['is_attributed'] == 0].sample(n=5_000_000 - len(legit), random_state=42)

df = pd.concat([legit, fraud_sample]).sort_values('click_time').reset_index(drop=True)

print(f'Undersampled dataset: {len(df):,} rows')
print(f'Legitimate: {(df["is_attributed"]==1).sum():,}')
print(f'Fraud:      {(df["is_attributed"]==0).sum():,}')
print(f'Fraud rate: {(df["is_attributed"]==0).mean()*100:.2f}%')


In [ ]:
print(f'Total rows:      {len(df):,}')
print(f'Legitimate (1):  {(df["is_attributed"] == 1).sum():,}')
print(f'Fraud proxy (0): {(df["is_attributed"] == 0).sum():,}')
print(f'Date range:      {df["click_time"].min()}  ->  {df["click_time"].max()}')

split_date = df['click_time'].max().floor('D')   # start of last day
train = df[df['click_time'] <  split_date].copy()
test  = df[df['click_time'] >= split_date].copy()
print(f'Train: {len(train):,} rows  |  Test: {len(test):,} rows')



In [ ]:
# All group features computed on train only, merged onto test
train = train.sort_values('click_time').reset_index(drop=True)
test  = test.sort_values('click_time').reset_index(drop=True)

# Temporal
for split in [train, test]:
    split['hour']     = split['click_time'].dt.hour.astype('uint8')
    split['is_night'] = ((split['hour'] >= 22) | (split['hour'] <= 5)).astype('uint8')

# Count / frequency — use MultiIndex map instead of merge
count_groups = [
    (['app'],                 'app_count'),
    (['ip', 'app'],           'ip_app_count'),
    (['ip', 'device'],        'ip_device_count'),
    (['ip', 'os'],            'ip_os_count'),
    (['ip', 'channel'],       'ip_channel_count'),
    (['ip', 'app', 'os'],     'ip_app_os_count'),
    (['ip', 'app', 'device'], 'ip_app_device_count'),
    (['app', 'channel'],      'app_channel_count'),
]
for group_cols, feat_name in count_groups:
    counts = train.groupby(group_cols).size().astype('int32')
    if len(group_cols) == 1:
        train[feat_name] = train[group_cols[0]].map(counts).astype('int32')
        test[feat_name]  = test[group_cols[0]].map(counts).fillna(0).astype('int32')
    else:
        idx_train = pd.MultiIndex.from_arrays([train[c] for c in group_cols])
        idx_test  = pd.MultiIndex.from_arrays([test[c]  for c in group_cols])
        train[feat_name] = idx_train.map(counts).astype('float32').fillna(0).astype('int32')
        test[feat_name]  = idx_test.map(counts).astype('float32').fillna(0).astype('int32')
print('Count features done.')

# Time deltas
td_groups = [['ip', 'app'], ['ip', 'device']]
for grp in td_groups:
    suffix = '_'.join(grp)
    for split in [train, test]:
        prev = split.groupby(grp)['click_time'].shift(1)
        nxt  = split.groupby(grp)['click_time'].shift(-1)
        split[f'prev_delta_{suffix}'] = (split['click_time'] - prev).dt.total_seconds().fillna(-1).astype('float32')
        split[f'next_delta_{suffix}'] = (nxt - split['click_time']).dt.total_seconds().fillna(-1).astype('float32')
print('Time delta features done.')

# Rolling window features
for part in [train, test]:
    part['bin_5min'] = part['click_time'].dt.floor('5min')
    part['bin_1h']   = part['click_time'].dt.floor('h')

rolling_groups = [
    (['ip', 'bin_5min'],          'ip_clicks_5min'),
    (['ip', 'app', 'bin_5min'],   'ip_app_clicks_5min'),
    (['ip', 'bin_1h'],            'ip_clicks_1h'),
    (['app', 'bin_5min'],         'app_clicks_5min'),
    (['ip', 'channel', 'bin_5min'], 'ip_channel_clicks_5min'),
]
for group_cols, feat_name in rolling_groups:
    counts = train.groupby(group_cols).size().astype('int32')
    idx_train = pd.MultiIndex.from_arrays([train[c] for c in group_cols])
    idx_test  = pd.MultiIndex.from_arrays([test[c]  for c in group_cols])
    train[feat_name] = idx_train.map(counts).astype('float32').fillna(0).astype('int32')
    test[feat_name]  = idx_test.map(counts).astype('float32').fillna(0).astype('int32')

train.drop(columns=['bin_5min', 'bin_1h'], inplace=True)
test.drop(columns=['bin_5min', 'bin_1h'],  inplace=True)
print('Rolling window features done.')

# Diversity
diversity = [
    ('ip',     'channel', 'ip_unique_channels'),
    ('ip',     'os',      'ip_unique_os'),
    ('device', 'ip',      'device_unique_ips'),
]
for group_col, target_col, feat_name in diversity:
    div = train.groupby(group_col)[target_col].nunique().astype('int32')
    train[feat_name] = train[group_col].map(div).astype('int32')
    test[feat_name]  = test[group_col].map(div).fillna(1).astype('int32')
print('Diversity features done.')

# Interval stats
valid = train[train['prev_delta_ip_app'] >= 0]
stats = valid.groupby(['ip', 'app'])['prev_delta_ip_app'].agg(
    ip_app_mean_interval='mean', ip_app_std_interval='std').fillna(0)
for feat_name in ['ip_app_mean_interval', 'ip_app_std_interval']:
    col = stats[feat_name]
    idx_train = pd.MultiIndex.from_arrays([train['ip'], train['app']])
    idx_test  = pd.MultiIndex.from_arrays([test['ip'],  test['app']])
    train[feat_name] = idx_train.map(col).fillna(0).astype('float32')
    test[feat_name]  = idx_test.map(col).fillna(0).astype('float32')
print('Interval stats done.')

# Conversion rates
global_rate = train['is_attributed'].mean()
for group_cols, feat_name in [
    (['app'],       'app_conv_rate'),
    (['channel'],   'channel_conv_rate'),
    (['ip', 'app'], 'ip_app_conv_rate'),
]:
    rates = train.groupby(group_cols)['is_attributed'].mean().astype('float32')
    if len(group_cols) == 1:
        train[feat_name] = train[group_cols[0]].map(rates).astype('float32')
        test[feat_name]  = test[group_cols[0]].map(rates).fillna(global_rate).astype('float32')
    else:
        idx_train = pd.MultiIndex.from_arrays([train[c] for c in group_cols])
        idx_test  = pd.MultiIndex.from_arrays([test[c]  for c in group_cols])
        train[feat_name] = idx_train.map(rates).fillna(global_rate).astype('float32')
        test[feat_name]  = idx_test.map(rates).fillna(global_rate).astype('float32')
print('Conversion rates done.')



In [ ]:
# Testing feature generation
sample_size = 200_000
ft_data = train.sample(n=sample_size, random_state=42).reset_index(drop=True)
ft_data = ft_data.reset_index().rename(columns={'index': 'click_id'})

# Drop target — don't want it generating "MEAN(is_attributed)" as a feature (leakage)
y_ft = ft_data['is_attributed']
ft_data = ft_data.drop(columns=['is_attributed', 'attributed_time'], errors='ignore')

# Build EntitySet — clicks as the main table, each categorical gets its own entity
es = ft.EntitySet(id='click_fraud')
es = es.add_dataframe(
    dataframe_name='clicks',
    dataframe=ft_data,
    index='click_id',
    time_index='click_time',
)
for col in ['ip', 'app', 'device', 'os', 'channel']:
    es = es.normalize_dataframe(
        base_dataframe_name='clicks',
        new_dataframe_name=col + 's',
        index=col,
    )
print(es)

# Run Deep Feature Synthesis
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name='clicks',
    agg_primitives=['count', 'mean', 'std', 'num_unique'],
    trans_primitives=[],
    max_depth=2,
    verbose=True,
)

print(f'Generated {len(feature_defs)} features')

# Light significance test — univariate AUC on the sample
from sklearn.metrics import roc_auc_score

fm = feature_matrix.fillna(0)
results = []
for col in fm.columns:
    try:
        auc = roc_auc_score(y_ft.loc[fm.index], fm[col])
        auc = max(auc, 1 - auc)
        results.append({'feature': col, 'univariate_auc': round(auc, 4)})
    except Exception:
        pass

auc_df = pd.DataFrame(results).set_index('feature').sort_values('univariate_auc', ascending=False)
shortlist = auc_df[auc_df['univariate_auc'] > 0.75].index.tolist()
print(f'\nShortlisted {len(shortlist)} features for MI test')

if len(shortlist) > 0:
    from sklearn.feature_selection import mutual_info_classif
    mi_scores = mutual_info_classif(fm[shortlist].sample(n=min(100_000, len(fm)), random_state=42),
                                    y_ft.loc[fm[shortlist].sample(n=min(100_000, len(fm)), random_state=42).index],
                                    random_state=42)
    mi_df = pd.DataFrame({'feature': shortlist, 'mutual_info': mi_scores}).set_index('feature')
    print(mi_df.sort_values('mutual_info', ascending=False).to_string())

print(f'\nTop 20 Featuretools features by univariate AUC:')
print(auc_df.head(20).to_string())


In [ ]:
# ALL generated features model
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import roc_auc_score


X_ft = feature_matrix.fillna(0)
y_ft_aligned = y_ft.loc[X_ft.index]

X_tr, X_val, y_tr, y_val = train_test_split(
    X_ft, y_ft_aligned, test_size=0.2, random_state=42
)

ft_model = lgb.LGBMClassifier(
    n_estimators=100, max_depth=4, num_leaves=15,
    n_jobs=-1, verbose=-1, random_state=42
)
print('Training on Featuretools features...')
ft_model.fit(X_tr, y_tr)

y_prob_ft = ft_model.predict_proba(X_val)[:, 1]
auc_ft    = roc_auc_score(y_val, y_prob_ft)
print(f'ROC-AUC: {auc_ft:.4f}')

imp = pd.Series(
    ft_model.booster_.feature_importance(importance_type='gain'),
    index=X_ft.columns
).sort_values(ascending=False)

print(f'\nTop 20 features by gain:')
print(imp.head(20).to_string())
print(f'\nDead features (0 gain): {(imp == 0).sum()} / {len(imp)}')


In [ ]:
# Separate non-zero features into existing vs genuinely new
feature_cols_all = [
    # Temporal
    'hour', 'is_night',
    # Count / frequency
    'app_count',
    'ip_app_count', 'ip_device_count', 'ip_os_count', 'ip_channel_count',
    'ip_app_os_count', 'ip_app_device_count', 'app_channel_count',
    # Time deltas
    'prev_delta_ip_app', 'next_delta_ip_app',
    'prev_delta_ip_device', 'next_delta_ip_device',
    # Diversity
    'ip_unique_channels', 'ip_unique_os', 'device_unique_ips',
    # Interval stats
    'ip_app_mean_interval', 'ip_app_std_interval',
    # Conversion rates
    'app_conv_rate', 'channel_conv_rate', 'ip_app_conv_rate',
    # Raw categoricals
    'app', 'channel',
    # Rolling window
    'ip_clicks_5min', 'ip_app_clicks_5min', 'ip_clicks_1h',
    'app_clicks_5min', 'ip_channel_clicks_5min',
]

nonzero = imp[imp > 0].index.tolist()
existing_ft = [f for f in nonzero if f in feature_cols_all]
new_ft      = [f for f in nonzero if f not in feature_cols_all]

print(f'Non-zero gain features: {len(nonzero)}')
print(f'Already in pipeline:    {len(existing_ft)}')
print(f'New from Featuretools:  {len(new_ft)}')
print('\nNew features to implement:')
for f in new_ft:
    print(f'  {f}')


In [ ]:
# Construct features on all training data
import re
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')


entity_map = {'ips': 'ip', 'apps': 'app', 'devices': 'device', 'oss': 'os', 'channels': 'channel'}
agg_map    = {'MEAN': 'mean', 'STD': 'std', 'MAX': 'max', 'MIN': 'min', 'NUM_UNIQUE': 'nunique'}

tasks = defaultdict(dict)

for feat in new_ft:
    m = re.match(r'(\w+)\.COUNT\(clicks\)$', feat)
    if m:
        group_col = entity_map.get(m.group(1))
        if group_col:
            tasks[group_col][f'ft_{group_col}_count'] = ('size', None)
        continue

    m = re.match(r'(\w+)\.(\w+)\(clicks\.(\w+)\)$', feat)
    if m:
        entity_plural, agg_name, col = m.groups()
        group_col = entity_map.get(entity_plural)
        agg_func  = agg_map.get(agg_name)
        if group_col and agg_func and col in train.columns:
            feat_name = f'ft_{entity_plural}_{agg_name.lower()}_{col}'
            tasks[group_col][feat_name] = (col, agg_func)

new_feature_names = []
for group_col, feats in tasks.items():
    agg_dict = {k: pd.NamedAgg(column=v[0], aggfunc=v[1])
                for k, v in feats.items() if v[0] != 'size'}
    size_feats = [k for k, v in feats.items() if v[0] == 'size']

    grp = train.groupby(group_col).agg(**agg_dict)
    if size_feats:
        grp[size_feats[0]] = train.groupby(group_col).size()

    for col in grp.columns:
        train[col] = train[group_col].map(grp[col]).astype('float32')
        test[col]  = test[group_col].map(grp[col]).fillna(0).astype('float32')
        new_feature_names.append(col)

    print(f'  ✓ {group_col}: {len(grp.columns)} features')

print(f'\nAdded {len(new_feature_names)} new features')



In [ ]:
# LGBM static (without 0 gain features)
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
feature_cols_all = [
    # Temporal
    'hour', 'is_night',
    # Count / frequency
    'app_count',
    'ip_app_count', 'ip_device_count', 'ip_os_count', 'ip_channel_count',
    'ip_app_os_count', 'ip_app_device_count', 'app_channel_count',
    # Time deltas
    'prev_delta_ip_app', 'next_delta_ip_app',
    'prev_delta_ip_device', 'next_delta_ip_device',
    # Diversity
    'ip_unique_channels', 'ip_unique_os', 'device_unique_ips',
    # Interval stats
    'ip_app_mean_interval', 'ip_app_std_interval',
    # Conversion rates
    'app_conv_rate', 'channel_conv_rate', 'ip_app_conv_rate',
    # Raw categoricals
    'app', 'channel',
    # Rolling window
    'ip_clicks_5min', 'ip_app_clicks_5min', 'ip_clicks_1h',
    'app_clicks_5min', 'ip_channel_clicks_5min',
]
feature_cols_all = feature_cols_all + new_feature_names
print(f'Total features: {len(feature_cols_all)}')


lgb_quick = lgb.LGBMClassifier(
    n_estimators=30,
    learning_rate=0.1,
    max_depth=6,
    num_leaves=15,
    min_child_samples=50,
    subsample=0.2,
    subsample_freq=1,
    colsample_bytree=0.4,
    is_unbalance=True, # added with 40 million rows vs 5 million
    n_jobs=-1, verbose=-1, random_state=42
)
print('Training quick LightGBM...')
lgb_quick.fit(train[feature_cols_all].fillna(0).astype('float32'), train['is_attributed'])

# Kaggle metric
y_prob = lgb_quick.predict_proba(test[feature_cols_all].fillna(0).astype('float32'))[:, 1]
local_auc = roc_auc_score(test['is_attributed'], y_prob)
print(f'\nLocal ROC-AUC (Kaggle metric): {local_auc:.4f}')
from sklearn.metrics import average_precision_score, confusion_matrix

y_pred = lgb_quick.predict(test[feature_cols_all].fillna(0)).astype('float32')
pr_auc = average_precision_score(test['is_attributed'], y_prob)
cm     = confusion_matrix(test['is_attributed'], y_pred)
tn, fp, fn, tp = cm.ravel()

print(f'PR-AUC (imbalanced metric):    {pr_auc:.4f}')
print(f'\nConfusion Matrix:')
print(f'  True  Negatives (correct fraud):   {tn:>10,}')
print(f'  False Positives (fraud missed):    {fp:>10,}')
print(f'  False Negatives (legit flagged):   {fn:>10,}')
print(f'  True  Positives (correct legit):   {tp:>10,}')
print(f'\nPrecision: {tp/(tp+fp):.4f}   Recall: {tp/(tp+fn):.4f}')


# Feature importance — fixed: gain and split are now different columns
importance_df = pd.DataFrame(index=feature_cols_all)

importance_df['lgbm_gain']  = pd.Series(
    lgb_quick.booster_.feature_importance(importance_type='gain'),  index=feature_cols_all)
importance_df['lgbm_split'] = pd.Series(
    lgb_quick.booster_.feature_importance(importance_type='split'), index=feature_cols_all)

print('\nDone.')
print(importance_df[['lgbm_gain', 'lgbm_split']].sort_values('lgbm_gain', ascending=False).to_string())

In [ ]:
# Early stopping LGBM model w/ full feature generation
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve


lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.3,
    subsample_freq=1,
    colsample_bytree=0.4,
    is_unbalance=True,
    n_jobs=-1, random_state=42, verbose=-1
)
print('Training LightGBM with early stopping...')
lgb_model.fit(
    train[feature_cols_all].fillna(0).astype('float32'), train['is_attributed'],
    eval_set=[(test[feature_cols_all].fillna(0).astype('float32'), test['is_attributed'])],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)]
)

y_prob = lgb_model.predict_proba(test[feature_cols_all].fillna(0).astype('float32'))[:, 1]
local_auc = roc_auc_score(test['is_attributed'], y_prob)
print(f'\nBest iteration: {lgb_model.best_iteration_}')
print(f'Local ROC-AUC (Kaggle metric): {local_auc:.4f}')

pr_auc = average_precision_score(test['is_attributed'], y_prob)
print(f'PR-AUC (imbalanced metric):    {pr_auc:.4f}')

# Feature importance
importance_df['lgbm_gain']  = pd.Series(
    lgb_model.booster_.feature_importance(importance_type='gain'),  index=feature_cols_all)
importance_df['lgbm_split'] = pd.Series(
    lgb_model.booster_.feature_importance(importance_type='split'), index=feature_cols_all)

print('\nDone.')
print(importance_df[['lgbm_gain', 'lgbm_split']].sort_values('lgbm_gain', ascending=False).to_string())


In [ ]:
# Mutual information of features
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score
from scipy.stats import pointbiserialr
import warnings
warnings.filterwarnings('ignore')

feature_cols_all = [
    # Temporal
    'hour', 'is_night',
    # Count / frequency
    'app_count',
    'ip_app_count', 'ip_device_count', 'ip_os_count', 'ip_channel_count',
    'ip_app_os_count', 'ip_app_device_count', 'app_channel_count',
    # Time deltas
    'prev_delta_ip_app', 'next_delta_ip_app',
    'prev_delta_ip_device', 'next_delta_ip_device',
    # Diversity
    'ip_unique_channels', 'ip_unique_os', 'device_unique_ips',
    # Interval stats
    'ip_app_mean_interval', 'ip_app_std_interval',
    # Conversion rates
    'app_conv_rate', 'channel_conv_rate', 'ip_app_conv_rate',
    # Raw categoricals
    'app', 'channel',
    # Rolling window
    'ip_clicks_5min', 'ip_app_clicks_5min', 'ip_clicks_1h',
    'app_clicks_5min', 'ip_channel_clicks_5min',
]
print(f'Total features: {len(feature_cols_all)}')

# Correlation + AUC: 5% sample (fast methods, scale fine)
sample   = train[feature_cols_all + ['is_attributed']].sample(n=500_000, random_state=42).fillna(0)
X_sample = sample[feature_cols_all]
y_sample = sample['is_attributed']
print(f'Sample size for correlation/AUC: {len(sample):,} rows')

# MI: cap at 100K — KNN-based estimator is slow at large n
mi_sample = sample.sample(n=min(100_000, len(sample)), random_state=42)
X_mi      = mi_sample[feature_cols_all]
y_mi      = mi_sample['is_attributed']
print(f'Sample size for MI:              {len(mi_sample):,} rows')

print('Running point-biserial correlation...')
results = []
for feat in feature_cols_all:
    pb_r, pb_p = pointbiserialr(y_sample, X_sample[feat])
    results.append({'feature': feat, 'pointbiserial_r': round(pb_r, 4), 'pb_p': round(pb_p, 4)})

print('Running mutual information...')
mi_scores = mutual_info_classif(X_mi, y_mi, random_state=42)

print('Running univariate AUC...')
auc_scores = [max(roc_auc_score(y_sample, X_sample[f]),
                  1 - roc_auc_score(y_sample, X_sample[f]))
              for f in feature_cols_all]

importance_df = pd.DataFrame(results).set_index('feature')
importance_df['mutual_info']    = mi_scores
importance_df['univariate_auc'] = auc_scores
importance_df = importance_df.sort_values('univariate_auc', ascending=False)

print('\nDone.\n')
print(importance_df[['pointbiserial_r', 'mutual_info', 'univariate_auc']].round(4).to_string())


In [ ]:
# Static LGBM model
import lightgbm as lgb
from sklearn.metrics import roc_auc_score


neg, pos = (train['is_attributed']==0).sum(), (train['is_attributed']==1).sum()

lgb_quick = lgb.LGBMClassifier(
    n_estimators=100, max_depth=6, num_leaves=63, min_child_samples=50,
    n_jobs=-1, verbose=-1, random_state=42
)
print('Training quick LightGBM...')
lgb_quick.fit(train[feature_cols_all].fillna(0), train['is_attributed'])

# Kaggle metric
y_prob    = lgb_quick.predict_proba(test[feature_cols_all].fillna(0))[:, 1]
local_auc = roc_auc_score(test['is_attributed'], y_prob)
print(f'\nLocal ROC-AUC (Kaggle metric): {local_auc:.4f}')
from sklearn.metrics import average_precision_score, confusion_matrix

y_pred = lgb_quick.predict(test[feature_cols_all].fillna(0))
pr_auc = average_precision_score(test['is_attributed'], y_prob)
cm     = confusion_matrix(test['is_attributed'], y_pred)
tn, fp, fn, tp = cm.ravel()

print(f'PR-AUC (imbalanced metric):    {pr_auc:.4f}')
print(f'\nConfusion Matrix:')
print(f'  True  Negatives (correct fraud):   {tn:>10,}')
print(f'  False Positives (fraud missed):    {fp:>10,}')
print(f'  False Negatives (legit flagged):   {fn:>10,}')
print(f'  True  Positives (correct legit):   {tp:>10,}')
print(f'\nPrecision: {tp/(tp+fp):.4f}   Recall: {tp/(tp+fn):.4f}')


# Feature importance — fixed: gain and split are now different columns
importance_df['lgbm_gain']  = pd.Series(
    lgb_quick.booster_.feature_importance(importance_type='gain'),  index=feature_cols_all)
importance_df['lgbm_split'] = pd.Series(
    lgb_quick.booster_.feature_importance(importance_type='split'), index=feature_cols_all)

print('\nDone.')
print(importance_df[['lgbm_gain', 'lgbm_split']].sort_values('lgbm_gain', ascending=False).to_string())


In [ ]:
# Early stopping LGBM model
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve

neg, pos = (train['is_attributed']==0).sum(), (train['is_attributed']==1).sum()

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=63,
    min_child_samples=100,
    colsample_bytree = .7,
    n_jobs=-1, random_state=42, verbose=-1
)
print('Training LightGBM with early stopping...')
lgb_model.fit(
    train[feature_cols_all].fillna(0), train['is_attributed'],
    eval_set=[(test[feature_cols_all].fillna(0), test['is_attributed'])],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)]
)

y_prob    = lgb_model.predict_proba(test[feature_cols_all].fillna(0))[:, 1]
local_auc = roc_auc_score(test['is_attributed'], y_prob)
print(f'\nBest iteration: {lgb_model.best_iteration_}')
print(f'Local ROC-AUC (Kaggle metric): {local_auc:.4f}')

pr_auc = average_precision_score(test['is_attributed'], y_prob)
print(f'PR-AUC (imbalanced metric):    {pr_auc:.4f}')

# Feature importance
importance_df['lgbm_gain']  = pd.Series(
    lgb_model.booster_.feature_importance(importance_type='gain'),  index=feature_cols_all)
importance_df['lgbm_split'] = pd.Series(
    lgb_model.booster_.feature_importance(importance_type='split'), index=feature_cols_all)

print('\nDone.')
print(importance_df[['lgbm_gain', 'lgbm_split']].sort_values('lgbm_gain', ascending=False).to_string())


In [ ]:
# LGBM x XGB ensemble testing
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

X_train = train[feature_cols_all].fillna(0)
y_train = train['is_attributed']
X_test  = test[feature_cols_all].fillna(0)
y_test  = test['is_attributed']

# XGBoost
print('Training XGBoost...')
xgb_model = XGBClassifier(
    n_estimators=40,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.2,
    colsample_bytree=0.4,
    min_child_weight=50,
    scale_pos_weight=1,
    tree_method='hist',
    n_jobs=-1, random_state=42,
    eval_metric='auc', verbosity=0
)
xgb_model.fit(
    train[feature_cols_all].astype('float32'), train['is_attributed']
)

# Predictions from each model
pred_lgb = lgb_model.predict_proba(X_test)[:, 1]
pred_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Individual scores
auc_lgb = roc_auc_score(y_test, pred_lgb)
auc_xgb = roc_auc_score(y_test, pred_xgb)
print(f'\nLightGBM AUC: {auc_lgb:.4f}')
print(f'XGBoost AUC:  {auc_xgb:.4f}')

# Simple average ensemble
pred_avg = (pred_lgb + pred_xgb) / 2
auc_avg  = roc_auc_score(y_test, pred_avg)
print(f'Average ensemble AUC: {auc_avg:.4f}')

# Weighted ensemble — weight by individual AUC
w_lgb = auc_lgb / (auc_lgb + auc_xgb)
w_xgb = auc_xgb / (auc_lgb + auc_xgb)
pred_weighted = w_lgb * pred_lgb + w_xgb * pred_xgb
auc_weighted  = roc_auc_score(y_test, pred_weighted)
print(f'Weighted ensemble AUC: {auc_weighted:.4f}  (lgb={w_lgb:.2f}, xgb={w_xgb:.2f})')


In [ ]:
# XGBoost
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

X_train_xgb = train[feature_cols_all].astype('float32')
X_test_xgb  = test[feature_cols_all].astype('float32')
y_train_xgb = train['is_attributed']
y_test_xgb  = test['is_attributed']

xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.2,
    colsample_bytree=0.4,
    min_child_weight=50,
    scale_pos_weight=1,
    tree_method='hist',       # faster than default
    n_jobs=-1, random_state=42,
    eval_metric='auc', verbosity=0,
    early_stopping_rounds=50
)

xgb_model.fit(
    X_train_xgb, y_train_xgb,
    eval_set=[(X_test_xgb, y_test_xgb)],
    verbose=50
)

xgb_pred = xgb_model.predict_proba(X_test_xgb)[:, 1]
xgb_auc = roc_auc_score(y_test_xgb, xgb_pred)
print(f"XGBoost AUC: {xgb_auc:.4f} (best iter: {xgb_model.best_iteration})")


In [ ]:
# Catboost
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

cb_model = CatBoostClassifier(
    iterations=55,
    learning_rate=0.1,
    depth=6,
    subsample=0.2,
    bootstrap_type='Bernoulli',
    rsm=0.4,
    min_data_in_leaf=50,
    class_weights=[1, 410],
    random_seed=42,
    verbose=25
)


cb_model.fit(
    train[feature_cols_all], train['is_attributed'],
    cat_features=['app', 'channel']
)

cb_pred = cb_model.predict_proba(test[feature_cols_all])[:, 1]
cb_auc = roc_auc_score(test['is_attributed'], cb_pred)
print(f"CatBoost AUC: {cb_auc:.4f}")


In [ ]:
# Ensemble with catboost
lgb_pred = lgb_model.predict_proba(test[feature_cols_all].astype('float32'))[:, 1]

auc_lgb = roc_auc_score(test['is_attributed'], lgb_pred)
auc_xgb = roc_auc_score(test['is_attributed'], xgb_pred)
auc_cb  = roc_auc_score(test['is_attributed'], cb_pred)
print(f'LightGBM AUC: {auc_lgb:.4f}')
print(f'XGBoost AUC:  {auc_xgb:.4f}')
print(f'CatBoost AUC: {auc_cb:.4f}')

from scipy.stats import rankdata
ensemble_pred = (rankdata(lgb_pred) + rankdata(xgb_pred)) / 2
ensemble_auc = roc_auc_score(test['is_attributed'], ensemble_pred)
print(f'\nEnsemble AUC: {ensemble_auc:.4f}')


In [ ]:
# ── Generate Kaggle submission (using LGBM x XGB ensemble)
import warnings
warnings.filterwarnings('ignore')

print('Loading Kaggle test set...')
dtypes_test = {
    'click_id': 'uint32',
    'ip':       'uint32',
    'app':      'uint16',
    'device':   'uint16',
    'os':       'uint16',
    'channel':  'uint16',
}
sub = pd.read_csv('test.csv', dtype=dtypes_test, parse_dates=['click_time'])
print(f'Test rows: {len(sub):,}')

# Temporal
sub['hour']     = sub['click_time'].dt.hour.astype('uint8')
sub['is_night'] = ((sub['hour'] >= 22) | (sub['hour'] <= 5)).astype('uint8')

# Sort for time deltas
sub = sub.sort_values('click_time').reset_index(drop=True)

# Count features
count_groups = [
    (['app'],                 'app_count'),
    (['ip', 'app'],           'ip_app_count'),
    (['ip', 'device'],        'ip_device_count'),
    (['ip', 'os'],            'ip_os_count'),
    (['ip', 'channel'],       'ip_channel_count'),
    (['ip', 'app', 'os'],     'ip_app_os_count'),
    (['ip', 'app', 'device'], 'ip_app_device_count'),
    (['app', 'channel'],      'app_channel_count'),
]
for group_cols, feat_name in count_groups:
    counts = train.groupby(group_cols).size().astype('int32')
    if len(group_cols) == 1:
        sub[feat_name] = sub[group_cols[0]].map(counts).fillna(0).astype('int32')
    else:
        idx = pd.MultiIndex.from_arrays([sub[c] for c in group_cols])
        sub[feat_name] = idx.map(counts).astype('float32').fillna(0).astype('int32')

# Time deltas
for grp in [['ip', 'app'], ['ip', 'device']]:
    suffix = '_'.join(grp)
    prev = sub.groupby(grp)['click_time'].shift(1)
    nxt  = sub.groupby(grp)['click_time'].shift(-1)
    sub[f'prev_delta_{suffix}'] = (sub['click_time'] - prev).dt.total_seconds().fillna(-1).astype('float32')
    sub[f'next_delta_{suffix}'] = (nxt - sub['click_time']).dt.total_seconds().fillna(-1).astype('float32')

# Diversity
for group_col, target_col, feat_name in [
    ('ip', 'channel', 'ip_unique_channels'),
    ('ip', 'os',      'ip_unique_os'),
    ('device', 'ip',  'device_unique_ips'),
]:
    div = train.groupby(group_col)[target_col].nunique().astype('int32')
    sub[feat_name] = sub[group_col].map(div).fillna(1).astype('int32')

# Interval stats
valid = train[train['prev_delta_ip_app'] >= 0]
stats = valid.groupby(['ip', 'app'])['prev_delta_ip_app'].agg(
    ip_app_mean_interval='mean', ip_app_std_interval='std').fillna(0)
for feat_name in ['ip_app_mean_interval', 'ip_app_std_interval']:
    idx = pd.MultiIndex.from_arrays([sub['ip'], sub['app']])
    sub[feat_name] = idx.map(stats[feat_name]).fillna(0).astype('float32')

# Conversion rates
global_rate = train['is_attributed'].mean()
for group_cols, feat_name in [
    (['app'],       'app_conv_rate'),
    (['channel'],   'channel_conv_rate'),
    (['ip', 'app'], 'ip_app_conv_rate'),
]:
    rates = train.groupby(group_cols)['is_attributed'].mean().astype('float32')
    if len(group_cols) == 1:
        sub[feat_name] = sub[group_cols[0]].map(rates).fillna(global_rate).astype('float32')
    else:
        idx = pd.MultiIndex.from_arrays([sub[c] for c in group_cols])
        sub[feat_name] = idx.map(rates).fillna(global_rate).astype('float32')

# Rolling windows (sub's own time buckets)
sub['bin_5min'] = sub['click_time'].dt.floor('5min')
sub['bin_1h']   = sub['click_time'].dt.floor('h')
for group_cols, feat_name in [
    (['ip', 'bin_5min'],            'ip_clicks_5min'),
    (['ip', 'app', 'bin_5min'],     'ip_app_clicks_5min'),
    (['ip', 'bin_1h'],              'ip_clicks_1h'),
    (['app', 'bin_5min'],           'app_clicks_5min'),
    (['ip', 'channel', 'bin_5min'], 'ip_channel_clicks_5min'),
]:
    counts = sub.groupby(group_cols).size().astype('int32')
    idx = pd.MultiIndex.from_arrays([sub[c] for c in group_cols])
    sub[feat_name] = idx.map(counts).fillna(0).astype('int32')
sub.drop(columns=['bin_5min', 'bin_1h'], inplace=True)

# FT features
for group_col, feats in tasks.items():
    agg_dict = {k: pd.NamedAgg(column=v[0], aggfunc=v[1])
                for k, v in feats.items() if v[0] != 'size'}
    size_feats = [k for k, v in feats.items() if v[0] == 'size']
    grp = train.groupby(group_col).agg(**agg_dict)
    if size_feats:
        grp[size_feats[0]] = train.groupby(group_col).size()
    for col in grp.columns:
        sub[col] = sub[group_col].map(grp[col]).fillna(0).astype('float32')

from scipy.stats import rankdata

print('Predicting...')
X_sub = sub[feature_cols_all].fillna(0).astype('float32')

pred_lgb_sub = lgb_model.predict_proba(X_sub)[:, 1]
pred_xgb_sub = xgb_model.predict_proba(X_sub)[:, 1]

# Rank ensemble — normalize to [0,1] for submission
n = len(pred_lgb_sub)
sub['is_attributed'] = (rankdata(pred_lgb_sub) + rankdata(pred_xgb_sub)) / (2 * n)

sub[['click_id', 'is_attributed']].to_csv('submission.csv', index=False)
print('Saved submission.csv')
print(sub[['click_id', 'is_attributed']].head())
